# Dialect identification: Arabic Transformers vs the TF-IDF baseline

تدريب AraBERT وMARBERT وCAMeLBERT على نفس تقسيم الـ baseline. الـ notebook بتحفظ metrics بس على Drive (من غير checkpoints).

In [ ]:
%pip install -q "transformers>=4.46" sentencepiece accelerate scikit-learn
import os, sys, subprocess, torch
assert torch.cuda.is_available(), 'فعّل GPU: Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

def run(*args):
    proc = subprocess.Popen([sys.executable, '-m', *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
    assert proc.wait() == 0, f'failed: {args}'

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/Arabic Dialect Identifier + Egyptian-to-MSA Normalizer'
assert os.path.isdir(ROOT), 'فولدر المشروع مش موجود على Drive'
os.chdir(ROOT); sys.path.insert(0, ROOT)

# لو الكود الجديد مش على Drive: ارفع project_update zip هنا وهيتفك فوق المشروع
if not os.path.exists('src/dialect_id/train_transformer.py'):
    import zipfile
    uploaded = files.upload()
    for name in uploaded:
        zipfile.ZipFile(name).extractall('/content/drive/MyDrive')
assert os.path.exists('src/dialect_id/train_transformer.py') and os.path.exists('data/processed/dialect.csv')

In [ ]:
# MAX_PER_CLASS = None يستخدم كل الداتا. لو الوقت ضيق جرّب 25000. القيمة 5000 بتوازن الكلاسات (ممكن تحسّن الـ recall للمصري).
MAX_PER_CLASS = None
for model in ['marbert', 'camelbert', 'arabert']:
    args = ['src.dialect_id.train_transformer', '--model', model, '--input', 'data/processed/dialect.csv']
    if MAX_PER_CLASS:
        args += ['--max-per-class', str(MAX_PER_CLASS)]
    run(*args)

In [ ]:
run('src.eval.summarize')
print(open('reports/RESULTS.md', encoding='utf-8').read())

لو اسم نموذج طلع غلط (Hugging Face model id)، هيظهر خطأ واضح من `from_pretrained`: غيّر الـ id في `MODELS` داخل `src/dialect_id/train_transformer.py`.